# Ecobici — 02. Modelo estocástico por estación (Etapa 2)

En este notebook se construye el modelo de cada estación de la subred, se confronta con los datos y se obtienen las medidas de falla que le importan al operador: estación vacía (viaje negado) y estación llena (usuario que no puede devolver).

- **Subred:** Buenavista (271-272, 273-274) y Reforma (014, 027, 018).
- **Periodo:** septiembre 2026, 19 días hábiles.
- **Insumos:** tasas por estación y hora, y resultados de las pruebas de supuestos del EDA (`01A_demanda.ipynb`).

## 1. ¿Qué representa el modelo?

Cada estación se modela por separado como un **proceso de nacimiento y muerte con capacidad finita (M/M/1/N)**. Los "clientes" son las bicis y los docks son la capacidad.

- **Estado:** número de bicis ancladas, de 0 a N (docks).
- **Devolución** (nacimiento): el estado sube una bici, con tasa λ(h), si hay dock libre.
- **Retiro** (muerte): el estado baja una bici, con tasa μ(h), si hay bici.

    0  ⇄  1  ⇄  2  ⇄  …  ⇄  N
    
       λ →         ← μ

- **Tasas por hora:** λ y μ cambian según la hora del día, pero son constantes dentro de cada hora. Se estimaron en el EDA.
- **Cómo se resuelve:** la estación arranca a las 5:00 con x₀ bicis. Con las ecuaciones de Kolmogorov hacia adelante se calcula, hora por hora, la probabilidad de tener 0, 1, …, N bicis. Cada hora parte de la distribución con que terminó la anterior.

**Por qué no en estado estable:** las tasas cambian mucho durante el día y el desbalance se acumula hora tras hora. Además, una solución estable no depende de x₀, que es justo lo que decide el operador.

In [2]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import expm
from scipy.optimize import brentq

DIR_TABLAS, DIR_FIG = Path("outputs/tablas"), Path("outputs/figuras")
SUBRED = {"271-272": "Buenavista", "273-274": "Buenavista", "014": "Reforma", "027": "Reforma", "018": "Reforma"}
ORDEN, HORAS = list(SUBRED), np.arange(5, 24)

df = pd.read_parquet("data/clean/trips_2026-09.parquet")
DOCKS = pd.DataFrame(json.loads(Path("data/raw/station_information.json").read_text())["data"]["stations"]) \
          .set_index("short_name")["capacity"]

dias = df.loc[df.es_habil, "fecha"].drop_duplicates().sort_values()
n_dias = len(dias)

# Retiros por fecha-hora de retiro; devoluciones por fecha-hora de arribo
m_ret = df.es_habil & df.en_horario & df.retiro_subred
m_dev = df.arribo_subred & df.arribo.dt.normalize().isin(dias) & df.hora_arribo.between(5, 23)
eventos = {
    "retiro": pd.DataFrame({"estacion": df.loc[m_ret, "Ciclo_Estacion_Retiro"].astype(str), "t": df.loc[m_ret, "retiro"]}),
    "devolucion": pd.DataFrame({"estacion": df.loc[m_dev, "Ciclo_EstacionArribo"].astype(str), "t": df.loc[m_dev, "arribo"]}),
}

# Conteos por intervalo de 15 min (rejilla completa: los intervalos sin viajes valen 0)
rejilla = pd.MultiIndex.from_product([ORDEN, dias, HORAS, range(4)], names=["estacion", "fecha", "hora", "cuarto"])
conteos = {
    k: e.groupby([e.estacion, e.t.dt.normalize().rename("fecha"), e.t.dt.hour.rename("hora"),
                  (e.t.dt.minute // 15).rename("cuarto")]).size().reindex(rejilla, fill_value=0)
    for k, e in eventos.items()
}

### 1.1 ¿Se justifica M/M/1/N?

Las dos "M" suponen que las devoluciones y los retiros llegan como procesos de Poisson. Se revisa con dos indicadores, en las horas con al menos 5 llegadas por hora:

- **Dispersión (varianza/media)** de los conteos por hora entre días: vale 1 si es Poisson. Si es mayor que 1, hay más variación de un día a otro de la que supone el modelo.
- **CV de los tiempos entre llegadas** dentro de la hora: vale 1 si son exponenciales. Se calcula solo en horas sin intervalos vacíos, para no mezclar la censura.

La "N" está dada por los docks de cada estación.

In [3]:
filas = []
for proceso, c in conteos.items():
    por_hora = c.groupby(["estacion", "fecha", "hora"]).sum()
    sin_hueco = c.groupby(["estacion", "fecha", "hora"]).min() > 0
    e = eventos[proceso].sort_values("t")
    e = e.assign(fecha=e.t.dt.normalize(), hora=e.t.dt.hour)
    e["gap"] = e.groupby(["estacion", "fecha", "hora"]).t.diff().dt.total_seconds()
    e = e[sin_hueco.reindex(pd.MultiIndex.from_frame(e[["estacion", "fecha", "hora"]])).to_numpy()]
    cv = e.groupby(["estacion", "fecha", "hora"]).gap.agg(lambda g: g.std() / g.mean() if g.count() > 4 else np.nan)
    for est in ORDEN:
        x = por_hora.loc[est].unstack("hora")
        D = (x.var() / x.mean())[x.mean() >= 5]
        filas.append({"estación": est, "proceso": proceso,
                      "dispersión var/media": D.median(), "CV tiempos entre llegadas": cv.loc[est].median()})

justificacion = pd.DataFrame(filas).round(2)
justificacion.to_csv(DIR_TABLAS / "justificacion_mm1n.csv", index=False)
justificacion

,estación,proceso,dispersión var/media,CV tiempos entre llegadas
0,271-272,retiro,2.48,1.07
1,273-274,retiro,7.58,1.00
2,014,retiro,2.02,1.00
3,027,retiro,2.11,0.98
4,018,retiro,1.25,0.97
5,271-272,devolucion,3.31,1.06
6,273-274,devolucion,3.78,1.06
7,014,devolucion,1.75,1.00
8,027,devolucion,1.44,0.98
9,018,devolucion,1.27,0.97


**Qué vemos:** dentro de cada hora las llegadas son Poisson: el CV de los tiempos entre llegadas está entre 0.97 y 1.07 en las 5 estaciones. Lo que cambia es la tasa de un día a otro (dispersión de 1.25 a 7.58, mayor en Buenavista). Por eso se mantiene M/M/1/N para describir un día típico, y la variación entre días se cubre con escenarios de demanda en la Etapa 4.

### 1.2 Tasas por hora

Para cada estación y hora: λ = devoluciones por hora y μ = retiros por hora, promedio de los días hábiles.

**Huecos.** En algunas horas la estación pasa vacía (o llena) y la gente que llega no queda registrada, así que la tasa observada se queda corta. Una hora se marca como hueco cuando tiene muchos más intervalos de 15 min sin llegadas de los que esperaría Poisson.

En esas horas **reemplazamos la tasa observada por una tasa estimada**, calculada solo con los intervalos donde hubo actividad (Poisson truncado en cero). Esta aproximación está pendiente de validar con el profesor.

In [5]:
def tasa_censurada(c15):
    """Tasa por hora usando solo los intervalos de 15 min con actividad (Poisson truncado en cero)."""
    media_pos = c15[c15 > 0].mean()
    if not np.isfinite(media_pos) or media_pos <= 1:
        return np.nan
    return 4 * brentq(lambda m: m / (1 - np.exp(-m)) - media_pos, 1e-6, 500)

filas = []
for proceso, c in conteos.items():
    for (est, h), g in c.groupby(["estacion", "hora"]):
        obs = g.sum() / n_dias
        hueco = (obs >= 8) and ((g == 0).mean() - np.exp(-obs / 4) >= 0.10)
        filas.append({"estacion": est, "hora": h, "proceso": proceso, "tasa_obs": obs,
                      "hueco": hueco, "tasa": tasa_censurada(g) if hueco else obs})

tasas = pd.DataFrame(filas)
tasas["docks"] = tasas.estacion.map(DOCKS)
tasas.to_csv(DIR_TABLAS / "tasas_modelo.csv", index=False)

tasas.loc[tasas.hueco, ["estacion", "hora", "proceso", "tasa_obs", "tasa"]] \
     .rename(columns={"tasa_obs": "tasa observada", "tasa": "tasa estimada"}).round(1)

,estacion,hora,proceso,tasa observada,tasa estimada
2,014,7,retiro,9.1,11.4
57,271-272,5,retiro,17.9,20.8
61,271-272,9,retiro,61.9,73.5
62,271-272,10,retiro,28.3,35.8
63,271-272,11,retiro,13.0,17.4
68,271-272,16,retiro,12.1,13.9
70,271-272,18,retiro,14.5,16.5
78,273-274,7,retiro,39.6,54.8
79,273-274,8,retiro,24.6,35.2
80,273-274,9,retiro,15.9,26.2


**Qué vemos:** hay 13 horas hueco. Casi todas son retiros en Buenavista en la mañana (273-274 de 7 a 10 h, 271-272 de 9 a 11 h), justo cuando la estación se vacía, y dos son devoluciones en la 273-274 a las 21 y 22 h (estación llena). En esas horas el modelo usa la tasa estimada; en las demás, la observada.

### 1.3 Funciones del modelo

- `tasas_estacion`: entrega λ y μ por hora y los docks de una estación (ya con la tasa estimada en las horas hueco).
- `generador`: matriz de transición de una hora (sube con λ, baja con μ).
- `resolver_dia`: aplica Kolmogorov hora por hora desde las 5:00 con x₀ bicis. Devuelve la probabilidad de tener cada número de bicis en cada minuto y los rechazos esperados por hora: gente sin bici (μ · P(vacía)) y gente sin dock (λ · P(llena)). `reinicios` representa la visita del camión en la Etapa 3.

In [6]:
def tasas_estacion(est):
    """λ por hora (devoluciones), μ por hora (retiros) y docks de una estación."""
    t = tasas[tasas.estacion == est].pivot(index="hora", columns="proceso", values="tasa").reindex(HORAS).fillna(0)
    return t["devolucion"].to_numpy(), t["retiro"].to_numpy(), int(DOCKS[est])

def generador(lam, mu, N):
    """Matriz Q de un proceso de nacimiento y muerte con capacidad N."""
    Q = np.diag(np.full(N, lam), 1) + np.diag(np.full(N, mu), -1)
    return Q - np.diag(Q.sum(axis=1))

def resolver_dia(lam_h, mu_h, N, x0, pasos_por_hora=60, reinicios=None):
    """Kolmogorov hacia adelante hora por hora, desde las 5:00 con x0 bicis.
    reinicios = {paso: nivel} para la visita del camión (Etapa 3)."""
    reinicios, dt = reinicios or {}, 1 / pasos_por_hora
    p = np.eye(N + 1)[x0]
    P, rech_ret, rech_dev, paso = [p], np.zeros(len(lam_h)), np.zeros(len(lam_h)), 0
    for k, (lam, mu) in enumerate(zip(lam_h, mu_h)):
        E = expm(generador(lam, mu, N) * dt)
        for _ in range(pasos_por_hora):
            if paso in reinicios:
                p = P[-1] = np.eye(N + 1)[reinicios[paso]]
            p_sig = p @ E
            rech_ret[k] += mu * (p[0] + p_sig[0]) / 2 * dt       # gente sin bici
            rech_dev[k] += lam * (p[N] + p_sig[N]) / 2 * dt      # gente sin dock
            p = p_sig
            P.append(p)
            paso += 1
    return {"P": np.array(P), "rech_ret": rech_ret, "rech_dev": rech_dev}